## Import packages

In [1]:
import pandas as pd
import numpy as np
import re
# Display Setting
from IPython.display import display
pd.options.display.max_colwidth=100
pd.options.display.float_format="{:.2f}".format
pd.set_option("display.max_columns", None)
import warnings
warnings.simplefilter('ignore')

# Exploratory data analysis
import matplotlib.pyplot as plt
import seaborn as sns
import plotly as pl
import chart_studio.plotly as py
import cufflinks as cf
from plotly.offline import download_plotlyjs,init_notebook_mode,plot,iplot
sns.set_theme(style='darkgrid')  # default style
import tensorflow as tf
np.set_printoptions(precision=3, suppress=True)  # improve float readability
from sklearn import datasets


# Data preprocessing
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import auc,roc_curve,classification_report,confusion_matrix,mean_absolute_error,mean_squared_error,root_mean_squared_error
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.svm import SVC
from sklearn.cluster import KMeans

from sklearn.decomposition import PCA
import string
from sklearn.feature_extraction.text import CountVectorizer,TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
%matplotlib inline

## Helper Function

In [2]:
def calculate_days(col):
    if pd.isna(col):
        return np.nan
    
    col = str(col).strip().lower()
    units = re.match(r"^\s*(\d+)\s*(year|years|yr|yrs|month|months|mo|mos|week|weeks|wk|wks|day|days)\s*$",col)
    if not units:
        return np.nan
    val = float(units.group(1)); 
    unit = units.group(2)
    if unit in {"year","years","yr","yrs"}:
        return val * 365
    if unit in {"month","months","mo","mos"}:
        return val * 30
    if unit in {"week","weeks","wk","wks"}:
        return val * 7
    if unit in {"day","days"}:
        return val
    return np.nan
    

## Read data

Import raw data

In [3]:
animal_intakes_raw_data = pd.read_csv("Austin_Animal_Center_Intakes.csv")
animal_outcomes_raw_data = pd.read_csv("Austin_Animal_Center_Outcomes.csv")

In [4]:
print("In the intake dataset, we have {} records with {} variables".format(*animal_intakes_raw_data.shape))
print("In the outcome dataset, we have {} records with {} variables".format(*animal_outcomes_raw_data.shape))

In the intake dataset, we have 173812 records with 12 variables
In the outcome dataset, we have 173775 records with 12 variables


In [5]:
animal_intakes_raw_data.head()

,Animal ID,Name,DateTime,MonthYear,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color
0,A521520,Nina,10/01/2013 07:51:00 AM,October 2013,Norht Ec in Austin (TX),Stray,Normal,Dog,Spayed Female,7 years,Border Terrier/Border Collie,White/Tan
1,A664235,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
2,A664236,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
3,A664237,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
4,A664233,Stevie,10/01/2013 08:53:00 AM,October 2013,7405 Springtime in Austin (TX),Stray,Injured,Dog,Intact Female,3 years,Pit Bull Mix,Blue/White


In [6]:
animal_intakes_raw_data['Intake Type'].value_counts()

Intake Type
Stray                 119160
Owner Surrender        35563
Public Assist          10432
Wildlife                6483
Abandoned               1910
Euthanasia Request       264
Name: count, dtype: int64

In [7]:
animal_intakes_raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173812 entries, 0 to 173811
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Animal ID         173812 non-null  object
 1   Name              123821 non-null  object
 2   DateTime          173812 non-null  object
 3   MonthYear         173812 non-null  object
 4   Found Location    173812 non-null  object
 5   Intake Type       173812 non-null  object
 6   Intake Condition  173812 non-null  object
 7   Animal Type       173812 non-null  object
 8   Sex upon Intake   173811 non-null  object
 9   Age upon Intake   173812 non-null  object
 10  Breed             173812 non-null  object
 11  Color             173812 non-null  object
dtypes: object(12)
memory usage: 15.9+ MB


In [8]:
animal_intakes_raw_data.describe()

,Animal ID,Name,DateTime,MonthYear,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color
count,173812,123821,173812,173812,173812,173812,173812,173812,173811,173812,173812,173812
unique,156287,29774,119722,140,70183,6,20,5,5,55,3006,661
top,A721033,Luna,09/23/2016 12:00:00 PM,June 2015,Austin (TX),Stray,Normal,Dog,Intact Male,1 year,Domestic Shorthair Mix,Black/White
freq,33,761,64,2189,31541,119160,147141,94608,58996,28294,33665,17976


In [9]:
animal_outcomes_raw_data.head()


,Animal ID,Date of Birth,Name,DateTime,MonthYear,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color
0,A668305,2012-12-01,NaN,2013-12-02T00:00:00-05:00,12-2013,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Brown/Yellow
1,A673335,2012-02-22,NaN,2014-02-22T00:00:00-05:00,02-2014,Euthanasia,Suffering,Other,Unknown,2 years,Raccoon,Black/Gray
2,A675999,2013-04-03,NaN,2014-04-07T00:00:00-05:00,04-2014,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Green
3,A679066,2014-04-16,NaN,2014-05-16T00:00:00-05:00,05-2014,NaN,NaN,Other,Unknown,4 weeks,Rabbit Sh,Brown
4,A680855,2014-05-25,NaN,2014-06-10T00:00:00-05:00,06-2014,Transfer,Partner,Bird,Unknown,2 weeks,Duck,Yellow/Black


In [10]:
animal_outcomes_raw_data['Outcome Type'].value_counts()

Outcome Type
Adoption           84598
Transfer           48689
Return to Owner    25691
Euthanasia         10833
Died                1672
Rto-Adopt           1241
Disposal             877
Missing               92
Relocate              29
Stolen                 5
Lost                   2
Name: count, dtype: int64

In [11]:
animal_outcomes_raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173775 entries, 0 to 173774
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Animal ID         173775 non-null  object
 1   Date of Birth     173775 non-null  object
 2   Name              123991 non-null  object
 3   DateTime          173775 non-null  object
 4   MonthYear         173775 non-null  object
 5   Outcome Type      173729 non-null  object
 6   Outcome Subtype   79660 non-null   object
 7   Animal Type       173775 non-null  object
 8   Sex upon Outcome  173774 non-null  object
 9   Age upon Outcome  173766 non-null  object
 10  Breed             173775 non-null  object
 11  Color             173775 non-null  object
dtypes: object(12)
memory usage: 15.9+ MB


## EDA 

## Data Preprocessing 

#### Data Cleaning

useful Example from assignment 5

In [12]:
# def read_data():
#     ''''''
#     # Read data
#     df = pd.read_csv(
#         "https://download.mlcc.google.com/mledu-datasets/flavors_of_cacao.csv",
#         sep=",",
#         encoding='latin-1'
#     )
    
#     return df


# def clean_data(df):
#     ''''''
#     # Set the output display to have one digit for decimal places and limit it to
#     # printing 15 rows.
#     pd.options.display.float_format = '{:.2f}'.format
#     pd.options.display.max_rows = 15
    
#     # Rename the columns.
#     df.columns = [
#         'maker', 'specific_origin', 'reference_number',
#         'review_date', 'cocoa_percent', 'maker_location',
#         'rating', 'bean_type', 'broad_origin'
#     ]

#     # df.dtypes

#     # Replace empty/null values with "Blend"
#     df['bean_type'] = df['bean_type'].fillna('Blend')

#     # Cast bean_type to string to remove leading 'u'
#     df['bean_type'] = df['bean_type'].astype(str)
#     df['cocoa_percent'] = df['cocoa_percent'].str.strip('%')
#     df['cocoa_percent'] = pd.to_numeric(df['cocoa_percent'])

#     # Correct spelling mistakes, and replace city with country name
#     df['maker_location'] = df['maker_location']\
#     .str.replace('Amsterdam', 'Holland')\
#     .str.replace('U.K.', 'England')\
#     .str.replace('Niacragua', 'Nicaragua')\
#     .str.replace('Domincan Republic', 'Dominican Republic')

#     # Adding this so that Holland and Netherlands map to the same country.
#     df['maker_location'] = df['maker_location']\
#     .str.replace('Holland', 'Netherlands')

#     def cleanup_spelling_abbrev(text):
#         replacements = [
#             ['-', ', '], ['/ ', ', '], ['/', ', '], ['\(', ', '], [' and', ', '], [' &', ', '], ['\)', ''],
#             ['Dom Rep|DR|Domin Rep|Dominican Rep,|Domincan Republic', 'Dominican Republic'],
#             ['Mad,|Mad$', 'Madagascar, '],
#             ['PNG', 'Papua New Guinea, '],
#             ['Guat,|Guat$', 'Guatemala, '],
#             ['Ven,|Ven$|Venez,|Venez$', 'Venezuela, '],
#             ['Ecu,|Ecu$|Ecuad,|Ecuad$', 'Ecuador, '],
#             ['Nic,|Nic$', 'Nicaragua, '],
#             ['Cost Rica', 'Costa Rica'],
#             ['Mex,|Mex$', 'Mexico, '],
#             ['Jam,|Jam$', 'Jamaica, '],
#             ['Haw,|Haw$', 'Hawaii, '],
#             ['Gre,|Gre$', 'Grenada, '],
#             ['Tri,|Tri$', 'Trinidad, '],
#             ['C Am', 'Central America'],
#             ['S America', 'South America'],
#             [', $', ''], [',  ', ', '], [', ,', ', '], ['\xa0', ' '],[',\s+', ','],
#             [' Bali', ',Bali']
#         ]
#         for i, j in replacements:
#             text = re.sub(i, j, text)
#         return text

#     df['specific_origin'] = df['specific_origin'].str.replace('.', '').apply(cleanup_spelling_abbrev)

#     # Cast specific_origin to string
#     df['specific_origin'] = df['specific_origin'].astype(str)

#     # Replace null-valued fields with the same value as for specific_origin
#     df['broad_origin'] = df['broad_origin'].fillna(df['specific_origin'])

#     # Clean up spelling mistakes and deal with abbreviations
#     df['broad_origin'] = df['broad_origin'].str.replace('.', '').apply(cleanup_spelling_abbrev)

#     # Change 'Trinitario, Criollo' to "Criollo, Trinitario"
#     # Check with df['bean_type'].unique()
#     df.loc[df['bean_type'].isin(['Trinitario, Criollo']),'bean_type'] = "Criollo, Trinitario"
#     # Confirm with df[df['bean_type'].isin(['Trinitario, Criollo'])]

#     # Fix chocolate maker names
#     df.loc[df['maker']=='Shattel','maker'] = 'Shattell'
#     df['maker'] = df['maker'].str.replace(u'Na\xef\xbf\xbdve','Naive')

#     return df


# df = clean_data(read_data())
# print('Shape of data', df.shape)
# df.head()

In [13]:
animal_intakes_data = animal_intakes_raw_data.copy()

# Check Intakes ID uniqueness
# animal_intakes_data['Readoption'] = animal_intakes_data.groupby(['Animal ID']).cumcount()
# animal_intakes_data['Animal ID'] = animal_intakes_data.apply(lambda x : f"{x['Animal ID']}_{x['Readoption']}" if x['Readoption'] > 0 else x['Animal ID'],axis=1)

# filter by animal type: dog, cat
animal_intakes_data = animal_intakes_data[animal_intakes_data['Animal Type'].isin(['Dog','Cat'])]

# drop readoption
animal_intakes_data = animal_intakes_data[animal_intakes_data.groupby(['Animal ID']).cumcount() == 0]

# Fill name as Unknown if no name
animal_intakes_data['Name'] = animal_intakes_data['Name'].fillna('Unknown')
animal_intakes_data['Sex upon Intake'] = animal_intakes_data['Sex upon Intake'].fillna('Unknown')
# Format intake date
animal_intakes_data['DateTime'] = animal_intakes_data['DateTime'].apply(lambda x : x[0:10])
animal_intakes_data.insert(2,'Intake Date',pd.to_datetime(animal_intakes_data['DateTime'],errors = "coerce").dt.date)
# animal_intakes_data['Intake Date'] = pd.to_datetime(animal_intakes_data['DateTime'],errors = "coerce").dt.date

# Transfer age to days
# animal_intakes_data['Age upon Intake (days)'] = animal_intakes_data['Age upon Intake'].apply(calculate_days)

# Drop duplicated columns
animal_intakes_data.drop(['DateTime','MonthYear','Age upon Intake'],axis=1,inplace=True)

In [14]:
animal_intakes_data

,Animal ID,Name,Intake Date,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Breed,Color
0,A521520,Nina,2013-10-01,Norht Ec in Austin (TX),Stray,Normal,Dog,Spayed Female,Border Terrier/Border Collie,White/Tan
1,A664235,Unknown,2013-10-01,Abia in Austin (TX),Stray,Normal,Cat,Unknown,Domestic Shorthair Mix,Orange/White
2,A664236,Unknown,2013-10-01,Abia in Austin (TX),Stray,Normal,Cat,Unknown,Domestic Shorthair Mix,Orange/White
3,A664237,Unknown,2013-10-01,Abia in Austin (TX),Stray,Normal,Cat,Unknown,Domestic Shorthair Mix,Orange/White
4,A664233,Stevie,2013-10-01,7405 Springtime in Austin (TX),Stray,Injured,Dog,Intact Female,Pit Bull Mix,Blue/White
...,...,...,...,...,...,...,...,...,...,...
173806,A929681,Unknown,2025-05-03,6816 Boyce Ln in Austin (TX),Stray,Neonatal,Cat,Intact Female,Domestic Shorthair,Brown Tabby
173807,A929690,Unknown,2025-05-03,8038 Exchange Dr in Austin (TX),Stray,Injured,Dog,Intact Male,Belgian Malinois,Brown/Black
173808,A929717,Unknown,2025-05-04,Austin (TX),Public Assist,Normal,Dog,Intact Male,Shih Tzu Mix,White/Blue
173810,A929725,Oswold,2025-05-04,1501 Red River St in Austin (TX),Public Assist,Normal,Dog,Intact Male,Boxer Mix,Tan/White


In [15]:
animal_outcomes_data = animal_outcomes_raw_data.copy()

# Check ID uniqueness
# animal_outcomes_data['Readoption'] = animal_outcomes_data.groupby(['Animal ID']).cumcount()
# animal_outcomes_data['Animal ID'] = animal_outcomes_data.apply(lambda x : f"{x['Animal ID']}_{x['Readoption']}" if x['Readoption'] > 0 else x['Animal ID'],axis=1)

# filter by animal type: dog, cat
animal_outcomes_data = animal_outcomes_data[animal_outcomes_data['Animal Type'].isin(['Dog','Cat'])]

# drop readoption
animal_outcomes_data[animal_outcomes_data.groupby(['Animal ID']).cumcount() == 0]

# change DOB dtype
animal_outcomes_data['Date of Birth'] = pd.to_datetime(animal_outcomes_data['Date of Birth'],errors = "coerce").dt.date

#  Format outcome date
animal_outcomes_data['DateTime'] = animal_outcomes_data['DateTime'].apply(lambda x : x[0:10])
animal_outcomes_data.insert(3,'Outcome Date',pd.to_datetime(animal_outcomes_data['DateTime'],errors = "coerce").dt.date)

# Fill Subtype as Unknown if no Subtype
animal_outcomes_data['Outcome Subtype'] = animal_outcomes_data['Outcome Subtype'].fillna('Unknown')

# Drop duplicated columns
animal_outcomes_data = animal_outcomes_data[['Animal ID','Outcome Date', 'Date of Birth', 'Outcome Type', 'Outcome Subtype']]

In [ ]:
# merge two dataset by animal ID
animal_data = pd.merge(animal_intakes_data,animal_outcomes_data,on='Animal ID',how='inner')

# remove missing value for Outcome Type
animal_data = animal_data[~animal_data['Outcome Type'].isnull()]

# Calcuate Length of Stay
animal_data["Length of Stay (days)"] = (animal_data['Outcome Date'] - animal_data['Intake Date']).apply(lambda x: x.days)

# Calcuate Age
animal_data.insert(8,'Age upon Intake (month)',((animal_data['Intake Date'] - animal_data['Date of Birth']).apply(lambda x: x.days))/30)

# remove wrong record
animal_data = animal_data[animal_data['Length of Stay (days)'] > 0]

In [21]:
animal_data.describe()

,Age upon Intake (month),Length of Stay (days)
count,146263.00,146263.00
mean,23.82,68.50
std,33.51,227.54
min,-51.83,1.00
25%,2.33,4.00
50%,12.17,10.00
75%,24.37,40.00
max,267.77,3884.00


In [22]:
animal_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 146263 entries, 7 to 163095
Data columns (total 16 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Animal ID                146263 non-null  object 
 1   Name                     146263 non-null  object 
 2   Intake Date              146263 non-null  object 
 3   Found Location           146263 non-null  object 
 4   Intake Type              146263 non-null  object 
 5   Intake Condition         146263 non-null  object 
 6   Animal Type              146263 non-null  object 
 7   Sex upon Intake          146263 non-null  object 
 8   Age upon Intake (month)  146263 non-null  float64
 9   Breed                    146263 non-null  object 
 10  Color                    146263 non-null  object 
 11  Outcome Date             146263 non-null  object 
 12  Date of Birth            146263 non-null  object 
 13  Outcome Type             146263 non-null  object 
 14  Outcome S

In [34]:
animal_data

,Animal ID,Name,Intake Date,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake (month),Breed,Color,Outcome Date,Date of Birth,Outcome Type,Outcome Subtype,Length of Stay (days)
7,A664256,*Donnie,2013-10-01,Austin (TX),Owner Surrender,Normal,Cat,Neutered Male,206.97,Domestic Shorthair Mix,Brown Tabby/White,2013-10-10,1996-10-01,Transfer,Partner,9
8,A664257,Pippin,2013-10-01,Burleson in Travis (TX),Stray,Normal,Dog,Intact Female,48.70,Podengo Pequeno Mix,Black,2013-10-24,2009-10-01,Adoption,Foster,23
9,A664266,Unknown,2013-10-01,Payton And 183 in Austin (TX),Stray,Normal,Dog,Intact Female,18.27,Chihuahua Shorthair Mix,Buff,2013-10-05,2012-04-01,Transfer,Partner,4
14,A651630,Heather,2013-10-01,Outside Jurisdiction,Owner Surrender,Injured,Dog,Spayed Female,78.77,Labrador Retriever/Chinese Sharpei,Tan,2013-12-31,2007-04-13,Adoption,Unknown,91
15,A664259,*Petunia,2013-10-01,5001 Creek Bend in Austin (TX),Stray,Normal,Dog,Intact Female,60.87,Pit Bull Mix,Black,2013-10-18,2008-10-01,Euthanasia,Aggressive,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163083,A929541,Unknown,2025-05-01,4814 Sunshine Drive in Austin (TX),Stray,Medical,Cat,Intact Female,1.00,Domestic Longhair,Blue/White,2025-05-04,2025-04-01,Euthanasia,Suffering,3
163084,A929559,Unknown,2025-05-01,10000 N Lamar Blvd in Austin (TX),Stray,Injured,Cat,Intact Male,1.43,Domestic Shorthair,Gray Tabby/White,2025-05-05,2025-03-19,Transfer,Partner,4
163085,A929560,Firulais,2025-05-01,1401 E Rundberg Ln in Austin (TX),Stray,Injured,Dog,Intact Male,73.07,Chihuahua Shorthair,Brown/White,2025-05-03,2019-05-01,Return to Owner,Unknown,2
163094,A929602,Unknown,2025-05-02,25204 Fawn Drive in Travis (TX),Stray,Normal,Cat,Unknown,12.17,Domestic Shorthair,Black,2025-05-03,2024-05-02,Transfer,Partner,1


In [36]:
animal_data['Intake Year'] = animal_data["Intake Date"].apply(lambda x: x.year if pd.notnull(x) else None)

In [ ]:
(animal_data['Intake Year'].value_counts()/len(animal_data['Intake Year'])).round(2)

Intake Year
2014   0.11
2019   0.11
2015   0.11
2016   0.10
2017   0.10
2018   0.10
2021   0.07
2022   0.07
2024   0.07
2023   0.06
2020   0.05
2013   0.03
2025   0.02
Name: count, dtype: float64

In [20]:
animal_data['Outcome Type'].value_counts()

Outcome Type
Adoption           82479
Transfer           37358
Return to Owner    20574
Euthanasia          3095
Rto-Adopt           1231
Died                1223
Disposal             209
Missing               83
Stolen                 5
Relocate               5
Lost                   1
Name: count, dtype: int64

In [24]:
""" 
Set up a for loop to iterate over each feature
Print the name of the feature (excel header)
Plot how many are empty/null/na/0
Print data type
If categorical:
Print top values, 
print unique values [na, 'missing', 'did not provide']
If numerical:
Print max, mean, plot distributions [ages may be off, duration may not make sense]
Put together a rough data dictionary, provide to stakeholders for feedback (share the cleanup load!)
Put together a correlation matrix
Spearman
Pearson
"""

" \nSet up a for loop to iterate over each feature\nPrint the name of the feature (excel header)\nPlot how many are empty/null/na/0\nPrint data type\nIf categorical:\nPrint top values, \nprint unique values [na, 'missing', 'did not provide']\nIf numerical:\nPrint max, mean, plot distributions [ages may be off, duration may not make sense]\nPut together a rough data dictionary, provide to stakeholders for feedback (share the cleanup load!)\nPut together a correlation matrix\nSpearman\nPearson\n"

In [25]:
col_dict = []
def col_info(df):
    for col in df:
        entry = {
            "feature":col,
            "dtype":str(df[col].dtype),
            "N/A values": df[col].isnull().sum(),
            "N/A values %": (df[col].isnull().mean()* 100).round(),
            "unique values":df[col].nunique(),
        }
        
        col_dict.append(entry)
    return col_dict

In [26]:
intake_col_info = col_info(animal_data)

#### Feature Engineering